# CMS Calorimeter Super-Resolution — Gated INR (v2, Optimised)
**GSoC 2026 | ML4Sci | Rajveer Rathod**

## What changed from v1
| # | Optimisation | Detail |
|---|---|---|
| 1 | **Inlined helpers** | All `calo_dataset.py` code lives in this notebook — zero `.py` dependency |
| 2 | **Mixed precision** | `torch.autocast('cuda')` + `GradScaler` on CUDA; skipped on MPS/CPU |
| 3 | **`zero_grad(set_to_none=True)`** | Avoids zeroing tensor memory; ~5% faster |
| 4 | **`non_blocking=True`** | All `.to(device)` transfers overlap with compute |
| 5 | **`torch.compile()`** | Applied on CUDA + PyTorch ≥ 2.0 (guarded try/except) |
| 6 | **`persistent_workers=True`, `prefetch_factor=4`** | Reduces worker spin-up overhead |
| 7 | **`pin_memory=True`** | Only on CUDA — avoids `cudaHostAlloc` on MPS |
| 8 | **`num_workers` auto-tune** | `min(8, cpu_count)` capped at 0 for MPS (macOS fork bug) |
| 9 | **Parallel cache build** | `ThreadPoolExecutor` for point-cloud conversion step |
| 10 | **`worker_init_fn`** | Reproducible per-worker RNG seeding |
| 11 | **Checkpoint robustness** | Saves/loads optimizer + scheduler + epoch + val_loss |
| 12 | **Deterministic seeding** | `cudnn.deterministic=True`, `benchmark=False` on CUDA |
| 13 | **Batched evaluation** | Groups samples into `BATCH_SIZE` chunks instead of one-at-a-time |
| 14 | **Memory cleanup** | `torch.cuda.empty_cache()` + `gc.collect()` after training |
| 15 | **Platform-agnostic paths** | No hardcoded local paths — all derived from `INPUT_DIR`/`OUTPUT_DIR` |


In [ ]:
# Safe to run on Colab / Kaggle / local — skips packages already present.
import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

try:
    import h5py
except ImportError:
    _pip('h5py')

try:
    import tqdm
except ImportError:
    _pip('tqdm')

try:
    import matplotlib
except ImportError:
    _pip('matplotlib')

print('Dependencies OK')


In [ ]:
import os, sys

# ── Detect runtime ────────────────────────────────────────────────────────────
def _detect_platform():
    if 'google.colab' in sys.modules or os.path.exists('/content'):
        return 'colab'
    if os.path.exists('/kaggle/input'):
        return 'kaggle'
    return 'local'

PLATFORM = _detect_platform()
print(f'Platform: {PLATFORM}')

if PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    # Adjust these paths to where your data lives on Drive
    INPUT_DIR  = '/content/drive/MyDrive/CaloChallenge'
    OUTPUT_DIR = '/content/drive/MyDrive/CaloSR_v2'
elif PLATFORM == 'kaggle':
    # Kaggle datasets are mounted read-only under /kaggle/input/
    INPUT_DIR  = '/kaggle/input/calochallenge'
    OUTPUT_DIR = '/kaggle/working'
else:
    # Local — edit these two lines for your machine
    INPUT_DIR  = os.path.expanduser(
        os.environ.get('CALO_INPUT_DIR',
                       '/Users/rajveerrathod/Work/Google_Summer_of_code/GAN/dataset'))
    OUTPUT_DIR = os.path.expanduser(
        os.environ.get('CALO_OUTPUT_DIR',
                       '/Users/rajveerrathod/Work/Google_Summer_of_code/GAN'))

# ── Derived directories ───────────────────────────────────────────────────────
CACHE_DIR   = os.path.join(OUTPUT_DIR, 'cache')
CKPT_DIR    = os.path.join(OUTPUT_DIR, 'trained_model', 'checkpoints')
FIGURES_ROOT = os.path.join(OUTPUT_DIR, 'figures')

for d in [CACHE_DIR, CKPT_DIR, FIGURES_ROOT]:
    os.makedirs(d, exist_ok=True)

# ── Per-run dated figure folder ───────────────────────────────────────────────
from datetime import datetime
RUN_TAG     = globals().get('RUN_TAG', datetime.now().strftime('%Y-%m-%d'))
FIGURES_DIR = os.path.join(FIGURES_ROOT, RUN_TAG)
os.makedirs(FIGURES_DIR, exist_ok=True)

print(f'INPUT_DIR  : {INPUT_DIR}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')
print(f'CACHE_DIR  : {CACHE_DIR}')
print(f'CKPT_DIR   : {CKPT_DIR}')
print(f'FIGURES_DIR: {FIGURES_DIR}')


def savefig(fig, name, dpi=150):
    """Save a matplotlib figure to the run's figures directory."""
    path = os.path.join(FIGURES_DIR, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  saved -> {path}')
    return path


In [ ]:
import gc, math, random, time, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm
import h5py
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

PIN_MEMORY = (DEVICE == 'cuda')

if DEVICE == 'cuda':
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    torch.cuda.manual_seed_all(SEED)

# MPS multiprocessing is buggy on macOS — cap at 0 workers
NUM_WORKERS = 0 if DEVICE == 'mps' else min(8, os.cpu_count() or 4)

# AMP: only meaningful on CUDA
USE_AMP  = (DEVICE == 'cuda')
SCALER   = torch.cuda.amp.GradScaler() if USE_AMP else None

print(f'Device      : {DEVICE}')
print(f'PyTorch     : {torch.__version__}')
print(f'NUM_WORKERS : {NUM_WORKERS}')
print(f'AMP enabled : {USE_AMP}')
if DEVICE == 'mps':
    print('Apple Silicon GPU (Metal Performance Shaders) — num_workers forced to 0')


In [ ]:
# ── Point-cloud helpers ───────────────────────────────────────────────────────
# Inlined from calo_dataset.py — vectorised np.add.at version (no Python loop).

def image_to_pointcloud(img: np.ndarray,
                        zero_suppression_threshold: float = 1e-6,
                        max_points: int = 1024) -> np.ndarray:
    """Convert (C, H, W) image -> (max_points, 2+C) point cloud.
    Coordinates normalised to [-1, 1].  Padding rows are all-zero."""
    C, H, W = img.shape
    eta_coords = np.linspace(-1, 1, H)
    phi_coords = np.linspace(-1, 1, W)
    ETA, PHI = np.meshgrid(eta_coords, phi_coords, indexing='ij')

    energy_sum = img.sum(axis=0)
    mask = energy_sum > zero_suppression_threshold

    eta_active = ETA[mask]
    phi_active = PHI[mask]
    e_active   = img[:, mask].T   # (N_active, C)

    points = np.concatenate([eta_active[:, None],
                              phi_active[:, None],
                              e_active], axis=-1)   # (N_active, 2+C)

    if len(points) >= max_points:
        order  = np.argsort(points[:, 2])[::-1][:max_points]
        points = np.ascontiguousarray(points[order])
    else:
        pad    = np.zeros((max_points - len(points), 2 + C), dtype=np.float32)
        points = np.concatenate([points, pad], axis=0)

    return np.ascontiguousarray(points, dtype=np.float32)


def pointcloud_to_image(points: np.ndarray,
                        target_H: int,
                        target_W: int,
                        C: int = 1,
                        aggregation: str = 'sum') -> np.ndarray:
    """Rasterise (N, 2+C) -> (C, H, W).  Vectorised scatter-add."""
    img   = np.zeros((C, target_H, target_W), dtype=np.float32)
    count = np.zeros((target_H, target_W),    dtype=np.float32)

    eta      = points[:, 0]
    phi      = points[:, 1]
    energies = points[:, 2:2 + C]

    i_idx = np.clip(np.round((eta + 1) / 2 * (target_H - 1)).astype(int), 0, target_H - 1)
    j_idx = np.clip(np.round((phi + 1) / 2 * (target_W - 1)).astype(int), 0, target_W - 1)

    # Vectorised — replaces the per-point Python loop (much faster).
    active = energies.sum(axis=1) != 0
    if active.any():
        flat = i_idx[active] * target_W + j_idx[active]
        for c in range(C):
            np.add.at(img[c].reshape(-1), flat, energies[active, c])
        np.add.at(count.reshape(-1), flat, 1.0)

    if aggregation == 'mean':
        m = count > 0
        img[:, m] /= count[m]

    return img


def apply_zero_suppression(img: np.ndarray, threshold: float) -> np.ndarray:
    """Zero out pixels below hardware threshold."""
    out = img.copy()
    out[out < threshold] = 0.0
    return out


def downsample_via_pointcloud(img: np.ndarray,
                               target_H: int,
                               target_W: int,
                               zero_thresh: float) -> np.ndarray:
    """Downsample (C,H,W) -> (C, target_H, target_W) via point-cloud rounding."""
    C, H, W = img.shape
    pc  = image_to_pointcloud(img, zero_suppression_threshold=zero_thresh,
                               max_points=H * W)
    out = pointcloud_to_image(pc, target_H, target_W, C=C, aggregation='sum')
    out = apply_zero_suppression(out, zero_thresh)
    return out


def pad_to_resolution(img: np.ndarray, target_H: int, target_W: int,
                      zero_thresh: float = 0.0) -> np.ndarray:
    """Resize (C,H,W) -> (C, target_H, target_W); downsample or center-pad."""
    C, H, W = img.shape
    if H == target_H and W == target_W:
        return img
    if H > target_H or W > target_W:
        return downsample_via_pointcloud(img, target_H, target_W, zero_thresh=zero_thresh)
    out = np.zeros((C, target_H, target_W), dtype=img.dtype)
    pad_h = (target_H - H) // 2
    pad_w = (target_W - W) // 2
    out[:, pad_h:pad_h + H, pad_w:pad_w + W] = img
    return out


print('Point-cloud helpers loaded (vectorised np.add.at version).')


In [ ]:
import json as _json
import concurrent.futures

# ── HDF5 index (streaming, never loads showers into RAM) ─────────────────────

def build_hdf5_index(paths, showers_key='showers', energy_key='incident_energies',
                     chunk=10_000):
    """
    Stream HDF5 files to build a row-level index WITHOUT loading showers into RAM.
    Returns: index (list of (path, local_row)), e_max (float), energies (N,1) float32.
    """
    index, energy_blocks, e_max = [], [], 0.0
    for path in paths:
        with h5py.File(path, 'r') as f:
            n = f[showers_key].shape[0]
            for r in range(n):
                index.append((path, r))
            for start in range(0, n, chunk):
                end   = min(start + chunk, n)
                block = f[showers_key][start:end]
                m     = float(block.max())
                if m > e_max:
                    e_max = m
                del block
            energy_blocks.append(f[energy_key][:].astype(np.float32))
    energies = np.concatenate(energy_blocks, axis=0)
    if e_max <= 0:
        e_max = 1.0
    return index, e_max, energies


def sample_raw_images(index, e_max, n_sample, cell_shape=(1, 45, 144),
                      showers_key='showers', seed=0):
    """Read n_sample random raw showers from the index for visualisation."""
    rng      = np.random.default_rng(seed)
    n_sample = min(n_sample, len(index))
    chosen   = rng.choice(len(index), n_sample, replace=False)
    out      = np.empty((n_sample, *cell_shape), dtype=np.float32)
    cache    = {}
    for k, gi in enumerate(chosen):
        path, row = index[gi]
        f = cache.get(path) or cache.setdefault(path, h5py.File(path, 'r'))
        out[k] = f[showers_key][row].reshape(cell_shape).astype(np.float32) / e_max
    for f in cache.values():
        f.close()
    return out, chosen


def build_image_cache(index, e_max, hr_res, lr_res, zero_thresh, cache_dir,
                      cell_shape=(1, 45, 144), showers_key='showers',
                      subset_n=None, seed=0, rebuild=False, num_proc=4):
    """
    Precompute HR+LR images and store them as memmapped .npy arrays.
    Cuts __getitem__ from ~31 ms to ~0.5 ms by moving point-cloud work offline.

    num_proc: number of threads for the pad/downsample step (HDF5 reads stay
              single-threaded for correctness; only the CPU-bound numpy work
              is parallelised).
    """
    os.makedirs(cache_dir, exist_ok=True)
    rng     = np.random.default_rng(seed)
    n_total = len(index)
    if subset_n is not None and subset_n < n_total:
        rows = np.sort(rng.choice(n_total, subset_n, replace=False))
    else:
        rows = np.arange(n_total)
    n = len(rows)

    C        = cell_shape[0]
    HR_H, HR_W = hr_res
    LR_H, LR_W = lr_res
    tag      = f'{n}_{HR_H}x{HR_W}_{LR_H}x{LR_W}_zt{zero_thresh}'
    hr_path  = os.path.join(cache_dir, f'hr_{tag}.npy')
    lr_path  = os.path.join(cache_dir, f'lr_{tag}.npy')
    beam_path = os.path.join(cache_dir, f'beam_{tag}.npy')
    meta_path = os.path.join(cache_dir, f'meta_{tag}.json')
    rows_path = os.path.join(cache_dir, f'rows_{tag}.npy')

    if (not rebuild and os.path.exists(hr_path) and os.path.exists(lr_path)
            and os.path.exists(meta_path) and os.path.exists(rows_path)):
        with open(meta_path) as f:
            meta = _json.load(f)
        print(f'[cache] reusing existing cache: {hr_path}  (n={meta["n"]})')
        meta['rows'] = np.load(rows_path)
        return meta

    print(f'[cache] building image cache for n={n} samples -> {cache_dir}')
    hr_mm = np.lib.format.open_memmap(hr_path, mode='w+', dtype=np.float32,
                                       shape=(n, C, HR_H, HR_W))
    lr_mm = np.lib.format.open_memmap(lr_path, mode='w+', dtype=np.float32,
                                       shape=(n, C, LR_H, LR_W))

    # Read HDF5 single-threaded (HDF5 is not thread-safe in general) then
    # parallelise the CPU-bound pad + downsample per sample.
    # Step 1: read all raw images sequentially into a buffer.
    print(f'  Step 1/2: reading {n} showers from HDF5 (sequential)...')
    raw_buf = np.empty((n, *cell_shape), dtype=np.float32)
    by_file = {}
    for k, gi in enumerate(rows):
        path, r = index[gi]
        by_file.setdefault(path, []).append((k, r))

    t0 = time.time()
    done = 0
    for path, items in by_file.items():
        with h5py.File(path, 'r') as f:
            dset = f[showers_key]
            for k, r in items:
                raw_buf[k] = dset[r].reshape(cell_shape).astype(np.float32) / e_max
                done += 1
                if done % 5000 == 0:
                    print(f'    read {done}/{n}  ({time.time()-t0:.0f}s)')

    # Step 2: parallel pad + downsample using ThreadPoolExecutor.
    print(f'  Step 2/2: computing HR+LR (parallel, num_proc={num_proc})...')

    def _process_sample(k):
        img = raw_buf[k]
        hr  = pad_to_resolution(img, HR_H, HR_W, zero_thresh=0.0)
        lr  = downsample_via_pointcloud(hr, LR_H, LR_W, zero_thresh)
        return k, hr, lr

    with concurrent.futures.ThreadPoolExecutor(max_workers=num_proc) as ex:
        futures = {ex.submit(_process_sample, k): k for k in range(n)}
        done2   = 0
        for fut in concurrent.futures.as_completed(futures):
            k, hr, lr   = fut.result()
            hr_mm[k]    = hr
            lr_mm[k]    = lr
            done2 += 1
            if done2 % 5000 == 0:
                print(f'    processed {done2}/{n}  ({time.time()-t0:.0f}s)')

    hr_mm.flush()
    lr_mm.flush()
    del raw_buf

    np.save(rows_path, rows)
    meta = dict(hr_path=hr_path, lr_path=lr_path, beam_path=beam_path,
                hr_res=list(hr_res), lr_res=list(lr_res),
                e_max=float(e_max), zero_thresh=float(zero_thresh),
                n=int(n), C=int(C))
    with open(meta_path, 'w') as f:
        _json.dump(meta, f)
    meta['rows'] = rows
    print(f'[cache] done in {time.time()-t0:.0f}s  ->  {hr_path}')
    return meta


print('HDF5 index + cache builder loaded.')


In [ ]:
class CalorimeterSRDataset(Dataset):
    """
    Per-sample output:
      lr_pc     (max_points, 2+C) float32  — LR point cloud (model input)
      hr_img    (C, HR_H, HR_W)   float32  — HR ground truth
      lr_img    (C, LR_H, LR_W)   float32  — LR image
      query_pts (n_query, 2)      float32  — (eta, phi) query coordinates
      query_e   (n_query, C)      float32  — HR energy at those pixels
      beam_e    scalar            float32  — incident beam energy

    Three backends (same output):
      cached  — memmapped HR+LR from build_image_cache (fastest, recommended)
      lazy    — per-row HDF5 read (minimal RAM; ~31 ms/sample)
      in-mem  — pass images directly as (N, C, H, W) array

    query oversampling: nonzero_frac of queries are drawn from active HR pixels
    so every sample carries real shower structure to learn (data is ~95% empty).
    """

    def __init__(self,
                 images: np.ndarray = None,
                 beam_energies: np.ndarray = None,
                 hr_res: tuple = (125, 125),
                 lr_res: tuple = (64, 64),
                 max_points: int = 512,
                 n_query: int = 512,
                 zero_thresh: float = 1e-3,
                 nonzero_frac: float = 0.5,
                 augment: bool = True,
                 precompute: bool = False,   # kept for API compat; ignored
                 index: list = None,
                 e_max: float = None,
                 cell_shape: tuple = (1, 45, 144),
                 showers_key: str = 'showers',
                 cache: dict = None,
                 cache_rows: np.ndarray = None):

        self.beam_energies = beam_energies
        self.hr_res        = hr_res
        self.lr_res        = lr_res
        self.max_points    = max_points
        self.n_query       = n_query
        self.zero_thresh   = zero_thresh
        self.nonzero_frac  = nonzero_frac
        self.augment       = augment

        self.images      = images
        self.index       = index
        self.e_max       = float(e_max) if e_max is not None else 1.0
        self.cell_shape  = cell_shape
        self.showers_key = showers_key
        self._cache_meta = cache
        self._cached     = cache is not None
        self._lazy       = (images is None) and not self._cached

        if self._cached:
            self._hr_path = cache['hr_path']
            self._lr_path = cache['lr_path']
            self.C        = cache['C']
            self._crows   = (cache_rows if cache_rows is not None
                             else np.arange(cache['n']))
            self._len     = len(self._crows)
            self._hr_mm   = None
            self._lr_mm   = None
        elif self._lazy:
            assert index is not None, 'lazy mode needs an index list'
            self.C            = cell_shape[0]
            self._len         = len(index)
            self._h5_handles  = {}
        else:
            _, C, H, W = images.shape
            self.C   = C
            self._len = len(images)

        # HR query grid shared across workers — tiny: (HR_H*HR_W, 2).
        HR_H, HR_W = hr_res
        eta = np.linspace(-1, 1, HR_H, dtype=np.float32)
        phi = np.linspace(-1, 1, HR_W, dtype=np.float32)
        ETA, PHI = np.meshgrid(eta, phi, indexing='ij')
        self._grid   = np.stack([ETA.ravel(), PHI.ravel()], axis=-1)
        self._n_full = HR_H * HR_W

    def __len__(self):
        return self._len

    def __getstate__(self):
        # h5py handles and memmaps cannot be pickled across workers.
        state = self.__dict__.copy()
        if '_h5_handles' in state:
            state['_h5_handles'] = {}
        state['_hr_mm'] = None
        state['_lr_mm'] = None
        return state

    def __setstate__(self, state):
        self.__dict__.update(state)
        if getattr(self, '_lazy', False):
            self._h5_handles = {}
        if getattr(self, '_cached', False):
            self._hr_mm = None
            self._lr_mm = None

    def _get_cache_mm(self):
        if self._hr_mm is None:
            self._hr_mm = np.load(self._hr_path, mmap_mode='r')
            self._lr_mm = np.load(self._lr_path, mmap_mode='r')
        return self._hr_mm, self._lr_mm

    def _sample_query_idx(self, hr_img):
        """Oversample active (non-zero) HR pixels so rare deposits are learned."""
        energy = hr_img.reshape(self.C, -1).sum(axis=0)
        active = np.flatnonzero(energy > 0)
        nq     = self.n_query
        n_pos  = int(round(nq * self.nonzero_frac))
        if len(active) == 0:
            n_pos = 0
        pos  = (np.random.choice(active, n_pos, replace=len(active) < n_pos)
                if n_pos > 0 else np.empty(0, dtype=np.int64))
        rest = np.random.choice(self._n_full, nq - n_pos, replace=False)
        return np.concatenate([pos, rest]).astype(np.int64)

    def _getitem_cached(self, idx):
        crow          = int(self._crows[idx])
        hr_mm, lr_mm  = self._get_cache_mm()
        hr_img        = np.array(hr_mm[crow], dtype=np.float32)
        lr_img        = np.array(lr_mm[crow], dtype=np.float32)

        grid = self._grid
        if self.augment and random.random() < 0.5:
            hr_img = hr_img[:, :, ::-1].copy()
            lr_img = lr_img[:, :, ::-1].copy()
            grid   = grid.copy()
            grid[:, 1] = -grid[:, 1]

        lr_pc     = image_to_pointcloud(lr_img, zero_suppression_threshold=self.zero_thresh,
                                        max_points=self.max_points)
        q_idx     = self._sample_query_idx(hr_img)
        query_pts = np.ascontiguousarray(grid[q_idx])
        query_e   = np.ascontiguousarray(hr_img.reshape(self.C, -1)[:, q_idx].T)

        return {
            'lr_pc':     torch.from_numpy(lr_pc),
            'hr_img':    torch.from_numpy(np.ascontiguousarray(hr_img)),
            'lr_img':    torch.from_numpy(np.ascontiguousarray(lr_img)),
            'query_pts': torch.from_numpy(query_pts),
            'query_e':   torch.from_numpy(query_e),
            'beam_e':    torch.tensor(float(self.beam_energies[idx, 0]), dtype=torch.float32),
        }

    def _get_h5(self, path):
        h = self._h5_handles.get(path)
        if h is None:
            h = h5py.File(path, 'r')
            self._h5_handles[path] = h
        return h

    def _raw_image(self, idx):
        if not self._lazy:
            return self.images[idx]
        path, row = self.index[idx]
        f = self._get_h5(path)
        shower = f[self.showers_key][row]
        return shower.reshape(self.cell_shape).astype(np.float32) / self.e_max

    def __getitem__(self, idx):
        if self._cached:
            return self._getitem_cached(idx)
        hr_img = pad_to_resolution(self._raw_image(idx), *self.hr_res, zero_thresh=0.0)
        hr_img = np.ascontiguousarray(hr_img, dtype=np.float32)

        grid = self._grid
        if self.augment and random.random() < 0.5:
            hr_img    = hr_img[:, :, ::-1].copy()
            grid      = grid.copy()
            grid[:, 1] = -grid[:, 1]

        lr_img = downsample_via_pointcloud(hr_img, *self.lr_res, self.zero_thresh)
        lr_pc  = image_to_pointcloud(lr_img, zero_suppression_threshold=self.zero_thresh,
                                     max_points=self.max_points)

        q_idx     = self._sample_query_idx(hr_img)
        query_pts = np.ascontiguousarray(grid[q_idx])
        hr_flat   = hr_img.reshape(self.C, -1)
        query_e   = np.ascontiguousarray(hr_flat[:, q_idx].T)

        return {
            'lr_pc':     torch.from_numpy(lr_pc),
            'hr_img':    torch.from_numpy(hr_img),
            'lr_img':    torch.from_numpy(lr_img),
            'query_pts': torch.from_numpy(query_pts),
            'query_e':   torch.from_numpy(query_e),
            'beam_e':    torch.tensor(float(self.beam_energies[idx, 0]), dtype=torch.float32),
        }


print('CalorimeterSRDataset defined.')


In [ ]:
# Walk INPUT_DIR and collect all HDF5 files.
hdf5_files = []
for root, dirs, files in os.walk(INPUT_DIR):
    for f in sorted(files):
        if f.endswith('.hdf5') or f.endswith('.h5'):
            hdf5_files.append(os.path.join(root, f))

if not hdf5_files:
    print(f'WARNING: No HDF5 files found in {INPUT_DIR}')
    print('Set INPUT_DIR (or CALO_INPUT_DIR env var) to the directory containing .h5 / .hdf5 files.')
else:
    print(f'Found {len(hdf5_files)} HDF5 file(s):')
    for p in hdf5_files:
        size_mb = os.path.getsize(p) / 1e6
        print(f'  {p}  ({size_mb:.1f} MB)')


In [ ]:
def explore_hdf5(path):
    """Recursively print HDF5 tree with shapes, dtypes, and value ranges."""
    with h5py.File(path, 'r') as f:
        def _walk(name, obj):
            pad = '  ' * (name.count('/') + 1)
            if isinstance(obj, h5py.Dataset):
                sample = obj[:min(5, obj.shape[0])]
                print(f'{pad}[dataset] {name}  shape={obj.shape}  dtype={obj.dtype}  '
                      f'min={float(np.min(sample)):.4f}  max={float(np.max(sample)):.4f}')
            else:
                print(f'{pad}[group]   {name}/')
        print(f'\n=== {os.path.basename(path)} ===')
        f.visititems(_walk)

for p in hdf5_files[:2]:   # inspect first two files (avoid huge output)
    explore_hdf5(p)


In [ ]:
CELL_SHAPE  = (1, 45, 144)   # (C, H, W) of one raw shower row
SHOWERS_KEY = 'showers'
SUBSET_N    = 30_000         # training+val pool. Set None for all 200k (slow).
ZERO_SUPPRESSION_THRESHOLD = 1e-3

RESOLUTION_LADDER = {
    'HR': (256, 256),
    'MR': (125, 125),
    'LR': (64,  64),
}
HR_RES = RESOLUTION_LADDER['MR']   # 125×125 — training target
LR_RES = RESOLUTION_LADDER['LR']   # 64×64   — model input

print('Building lazy HDF5 index (streaming, no dense load)...')
hdf5_index, E_MAX, energies_all = build_hdf5_index(
    hdf5_files, showers_key=SHOWERS_KEY, energy_key='incident_energies', chunk=10_000)
print(f'Indexed {len(hdf5_index):,} showers  |  E_MAX={E_MAX:.4f}')

# Build or reuse memmapped cache.
cache_meta = build_image_cache(
    hdf5_index, E_MAX, HR_RES, LR_RES, ZERO_SUPPRESSION_THRESHOLD,
    cache_dir=CACHE_DIR, cell_shape=CELL_SHAPE, showers_key=SHOWERS_KEY,
    subset_n=SUBSET_N, seed=SEED, rebuild=False, num_proc=4)

cache_rows   = cache_meta['rows']
energies_raw = energies_all[cache_rows]   # aligned beam energies
N = len(cache_rows)
C = CELL_SHAPE[0]

print('=' * 70)
print(f'Cache ready: N={N:,}  HR={HR_RES}  LR={LR_RES}  dir={CACHE_DIR}')
print('Resident RAM for showers: 0 bytes (memmapped from disk)')


In [ ]:
hr_mm = np.load(cache_meta['hr_path'], mmap_mode='r')
lr_mm = np.load(cache_meta['lr_path'], mmap_mode='r')

N_SHOW = 8
sel    = np.random.choice(N, N_SHOW, replace=False)

fig, axes = plt.subplots(2, N_SHOW // 2, figsize=(16, 6))
fig.suptitle('Sample CMS Calorimeter Jet Images (HR cache, log scale)', fontsize=13)
for i, ax in enumerate(axes.flat):
    img = np.clip(hr_mm[sel[i], 0], 1e-6, None)
    im  = ax.imshow(img, cmap='hot', norm=LogNorm(vmin=1e-4, vmax=1.0), aspect='auto')
    ax.set_title(f'jet {sel[i]}  E={energies_raw[sel[i], 0]:.1f}', fontsize=8)
    ax.axis('off')
fig.colorbar(im, ax=axes, label='E / E_MAX (log)')
plt.tight_layout()
savefig(fig, 'sample_jets')
plt.show()

# Distributions over a memmapped subset (no full load).
stat_n  = min(2000, N)
ss      = np.sort(np.random.choice(N, stat_n, replace=False))
hr_sub  = np.asarray(hr_mm[ss])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist((hr_sub * E_MAX).sum(axis=(1, 2, 3)), bins=50,
         color='#e84393', edgecolor='k', linewidth=0.4)
ax1.set_xlabel('Total Jet Energy (raw)'); ax1.set_ylabel('Count')
ax1.set_title(f'Energy Distribution (n={stat_n})')

ax2.hist((hr_sub > 0).sum(axis=(1, 2, 3)), bins=50,
         color='#00b4d8', edgecolor='k', linewidth=0.4)
ax2.set_xlabel('Active Pixels per Jet'); ax2.set_ylabel('Count')
ax2.set_title(f'Active Cell Count (n={stat_n})')

plt.tight_layout()
savefig(fig, 'energy_distributions')
plt.show()


In [ ]:
# Sample one raw shower for the ladder demo.
_demo, _ = sample_raw_images(hdf5_index, E_MAX, 1, cell_shape=CELL_SHAPE,
                              showers_key=SHOWERS_KEY, seed=7)
sample_raw = _demo[0]   # (C, 45, 144)

HR_H, HR_W = RESOLUTION_LADDER['HR']
sample_hr   = pad_to_resolution(sample_raw, HR_H, HR_W, zero_thresh=0.0)
sample_mr   = downsample_via_pointcloud(sample_hr, *RESOLUTION_LADDER['MR'],
                                        zero_thresh=ZERO_SUPPRESSION_THRESHOLD)
sample_lr   = downsample_via_pointcloud(sample_hr, *RESOLUTION_LADDER['LR'],
                                        zero_thresh=ZERO_SUPPRESSION_THRESHOLD)

print('Resolution Ladder:')
for name, arr in [('HR (256x256)', sample_hr), ('MR (125x125)', sample_mr),
                  ('LR (64x64)',   sample_lr)]:
    sparsity = (arr == 0).mean()
    print(f'  {name}  E={arr.sum():.4f}  sparsity={sparsity:.2%}')

scales = {'HR 256x256': sample_hr, 'MR 125x125': sample_mr, 'LR 64x64': sample_lr}
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Multi-Scale Downsampling Ladder (Point-Cloud Rounding + Zero Suppression)',
             fontsize=12)
for ax, (title, arr) in zip(axes, scales.items()):
    display = np.clip(arr[0], 1e-6, None)
    ax.imshow(display, cmap='hot', norm=LogNorm(vmin=1e-4, vmax=1.0), aspect='equal')
    e_resp = arr.sum() / (sample_hr.sum() + 1e-8)
    ax.set_title(f'{title}\nE response: {e_resp:.3f}  sparsity: {(arr==0).mean():.1%}',
                 fontsize=9)
    ax.axis('off')
plt.tight_layout()
savefig(fig, 'resolution_ladder')
plt.show()

# Energy response vs resolution.
resolutions = [256, 192, 125, 96, 64, 32, 16]
e_responses = []
for r in resolutions:
    ds = downsample_via_pointcloud(sample_hr, r, r,
                                   zero_thresh=ZERO_SUPPRESSION_THRESHOLD)
    e_responses.append(ds.sum() / (sample_hr.sum() + 1e-8))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(resolutions, e_responses, 'o-', color='#e84393', lw=2, ms=8)
ax.axhline(1.0, color='gray', ls='--', label='Ideal (no energy loss)')
ax.set_xlabel('Resolution (pixels per side)'); ax.set_ylabel('Energy Response')
ax.set_title('Energy Loss vs Resolution'); ax.legend(); ax.invert_xaxis()
plt.tight_layout()
savefig(fig, 'energy_vs_resolution')
plt.show()


In [ ]:
from mpl_toolkits.mplot3d import Axes3D   # noqa: F401
import matplotlib.cm as cm

hr_mm    = np.load(cache_meta['hr_path'], mmap_mode='r')
N_JETS   = 4
jet_idxs = np.random.choice(N, N_JETS, replace=False)

fig = plt.figure(figsize=(20, 5 * N_JETS))
fig.suptitle('3D Calorimeter Point Clouds  (eta, phi, E)', fontsize=14, fontweight='bold')

for row, ji in enumerate(jet_idxs):
    pc     = image_to_pointcloud(np.asarray(hr_mm[ji]), max_points=1024)
    active = pc[pc[:, 2] > 0]
    eta, phi, E = active[:, 0], active[:, 1], active[:, 2]
    E_norm  = (E - E.min()) / (E.max() - E.min() + 1e-9)
    sizes   = 5 + 80 * E_norm

    # 3D scatter
    ax3d = fig.add_subplot(N_JETS, 2, row * 2 + 1, projection='3d')
    sc   = ax3d.scatter(eta, phi, E, c=E, cmap='hot', s=sizes, alpha=0.85,
                        depthshade=True,
                        norm=LogNorm(vmin=E.min() + 1e-6, vmax=E.max()))
    ax3d.set_xlabel('eta', labelpad=6); ax3d.set_ylabel('phi', labelpad=6)
    ax3d.set_zlabel('E / E_MAX', labelpad=6)
    ax3d.set_title(f'jet {ji}  |  {len(active)} cells  |  E_beam={energies_raw[ji,0]:.0f}',
                   fontsize=9)
    ax3d.view_init(elev=25, azim=-60)
    fig.colorbar(sc, ax=ax3d, shrink=0.5, pad=0.1, label='E / E_MAX')

    # Top-down (eta-phi)
    ax2d = fig.add_subplot(N_JETS, 2, row * 2 + 2)
    sc2  = ax2d.scatter(eta, phi, c=E, cmap='hot', s=sizes * 0.6, alpha=0.85,
                        norm=LogNorm(vmin=E.min() + 1e-6, vmax=E.max()))
    ax2d.set_xlabel('eta (norm.)'); ax2d.set_ylabel('phi (norm.)')
    ax2d.set_title('Top-down projection (eta-phi plane)', fontsize=9)
    ax2d.set_facecolor('#0a0a0a')
    ax2d.set_xlim(-1, 1); ax2d.set_ylim(-1, 1)
    fig.colorbar(sc2, ax=ax2d, label='E / E_MAX')

plt.tight_layout()
savefig(fig, 'pointcloud_3d')
plt.show()

# Grid vs point cloud side-by-side for 4 jets.
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Grid Image vs Point Cloud Representation', fontsize=13)
for col in range(4):
    img_3d = np.asarray(hr_mm[col])
    img    = img_3d[0]
    pc     = image_to_pointcloud(img_3d, max_points=2048)
    active = pc[pc[:, 2] > 0]

    axes[0, col].imshow(np.clip(img, 1e-6, None), cmap='hot',
                        norm=LogNorm(vmin=1e-4, vmax=1.0), aspect='equal')
    axes[0, col].set_title(f'Grid {img.shape[0]}x{img.shape[1]}', fontsize=8)
    axes[0, col].axis('off')

    sc = axes[1, col].scatter(active[:, 1], active[:, 0], c=active[:, 2],
                               cmap='hot', s=8,
                               norm=LogNorm(vmin=1e-4, vmax=1.0))
    axes[1, col].set_xlim(-1, 1); axes[1, col].set_ylim(-1, 1)
    axes[1, col].set_title(f'Point cloud ({len(active)} pts)', fontsize=8)
    axes[1, col].set_xlabel('phi'); axes[1, col].set_ylabel('eta')
    axes[1, col].invert_yaxis()

plt.tight_layout()
savefig(fig, 'grid_vs_pointcloud')
plt.show()


In [ ]:
# ── Worker seeding for reproducible per-worker RNG ───────────────────────────
def _worker_init_fn(worker_id):
    seed = SEED + worker_id
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

# ── Train / val split ────────────────────────────────────────────────────────
TRAIN_FRAC = 0.85
N_TRAIN    = int(N * TRAIN_FRAC)
perm       = np.random.permutation(N)
train_pos, val_pos = perm[:N_TRAIN], perm[N_TRAIN:]

common = dict(hr_res=HR_RES, lr_res=LR_RES, max_points=512, n_query=512,
              nonzero_frac=0.3, zero_thresh=ZERO_SUPPRESSION_THRESHOLD,
              cache=cache_meta)
train_dataset = CalorimeterSRDataset(beam_energies=energies_raw[train_pos],
                                     cache_rows=train_pos, augment=True,  **common)
val_dataset   = CalorimeterSRDataset(beam_energies=energies_raw[val_pos],
                                     cache_rows=val_pos,   augment=False, **common)

BATCH_SIZE = 64
loader_kwargs = dict(
    batch_size   = BATCH_SIZE,
    num_workers  = NUM_WORKERS,
    pin_memory   = PIN_MEMORY,
    worker_init_fn = _worker_init_fn,
)
if NUM_WORKERS > 0:
    loader_kwargs.update(persistent_workers=True, prefetch_factor=4)

train_loader = DataLoader(train_dataset, shuffle=True,  **loader_kwargs)
val_loader   = DataLoader(val_dataset,   shuffle=False, **loader_kwargs)

print(f'Train: {len(train_dataset):,} samples  |  Val: {len(val_dataset):,} samples')
print(f'DataLoader: batch={BATCH_SIZE}, workers={NUM_WORKERS}, '
      f'{len(train_loader)} train / {len(val_loader)} val batches/epoch')
print(f'pin_memory={PIN_MEMORY}  persistent_workers={NUM_WORKERS > 0}  prefetch_factor=4')

# Quick smoke-test.
batch = next(iter(train_loader))
for k, v in batch.items():
    print(f'  {k:12s}  {str(tuple(v.shape)):30s}  dtype={v.dtype}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Model architecture
# ─────────────────────────────────────────────────────────────────────────────

class PointNetEncoder(nn.Module):
    """Lightweight PointNet: (B, N, 2+C) -> (B, latent_dim).
    Permutation- and resolution-invariant via global max-pool."""

    def __init__(self, in_features: int, latent_dim: int = 256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Conv1d(in_features, 64,  1), nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Conv1d(64,  128, 1),          nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 256, 1),          nn.BatchNorm1d(256), nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(256, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Linear(256, latent_dim),
        )

    def forward(self, pc: torch.Tensor) -> torch.Tensor:
        # pc: (B, N, 2+C) -> (B, latent_dim)
        x = pc.transpose(1, 2)        # (B, 2+C, N)
        x = self.mlp(x)               # (B, 256, N)
        x = x.max(dim=-1).values      # global max pool -> (B, 256)
        return self.head(x)


class SineLayer(nn.Module):
    """SIREN layer: y = sin(omega_0 * Wx + b).  Initialised per Sitzmann et al."""

    def __init__(self, in_f: int, out_f: int, omega_0: float = 30.0,
                 is_first: bool = False):
        super().__init__()
        self.omega_0 = omega_0
        self.linear  = nn.Linear(in_f, out_f)
        with torch.no_grad():
            if is_first:
                self.linear.weight.uniform_(-1 / in_f, 1 / in_f)
            else:
                b = math.sqrt(6 / in_f) / omega_0
                self.linear.weight.uniform_(-b, b)

    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))


class INRDecoder(nn.Module):
    """
    Gated decoder for SPARSE calorimeter data.

    Two independent pathways:
      MAGNITUDE : SIREN trunk -> softplus         (smooth energy value)
      OCCUPANCY : separate ReLU-MLP + Fourier PE  ('is there a hit?')
      output    : sigmoid(occ_logit) * magnitude

    The decoupled trunks prevent the occupancy head from stalling at ln2
    (which caused the 'black output' collapse seen when sharing the SIREN).
    """

    def __init__(self, latent_dim: int = 256, hidden_dim: int = 256,
                 n_layers: int = 5, out_C: int = 1, omega_0: float = 30.0):
        super().__init__()
        self.out_C   = out_C
        self.first   = SineLayer(2 + latent_dim, hidden_dim, omega_0, is_first=True)
        self.hidden  = nn.ModuleList([
            SineLayer(hidden_dim, hidden_dim, omega_0) for _ in range(n_layers - 2)
        ])
        self.mag_head = nn.Linear(hidden_dim, out_C)
        nn.init.constant_(self.mag_head.bias, 0.0)

        # Fourier positional encoding for the occupancy ReLU network.
        # Raw 2D coords through ReLU underfit sharp spatial boundaries —
        # Fourier features give the occ-net high-frequency capacity cheaply.
        self.n_fourier = 8
        freqs = 2.0 ** torch.arange(self.n_fourier).float() * math.pi
        self.register_buffer('occ_freqs', freqs)
        occ_in = latent_dim + 2 + 4 * self.n_fourier   # latent + raw + sin/cos
        self.occ_net = nn.Sequential(
            nn.Linear(occ_in, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, out_C),
        )
        nn.init.constant_(self.occ_net[-1].bias, -2.0)   # bias toward 'empty'

    def _occ_features(self, query, z_exp):
        # query: (B, N_q, 2);  z_exp: (B, N_q, latent)
        proj = query.unsqueeze(-1) * self.occ_freqs       # (B, N_q, 2, n_fourier)
        proj = proj.flatten(-2)                           # (B, N_q, 2*n_fourier)
        ff   = torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)  # (B,N_q,4*n_f)
        return torch.cat([query, ff, z_exp], dim=-1)

    def forward(self, z, query, return_parts: bool = False):
        B, N_q, _ = query.shape
        z_exp  = z.unsqueeze(1).expand(-1, N_q, -1)
        inp    = torch.cat([query, z_exp], dim=-1)   # (B, N_q, 2+latent)

        x = self.first(inp)
        for layer in self.hidden:
            x = layer(x)
        mag = F.softplus(self.mag_head(x))            # (B, N_q, C)

        occ_logit = self.occ_net(self._occ_features(query, z_exp))  # (B, N_q, C)
        energy    = torch.sigmoid(occ_logit) * mag
        if return_parts:
            return energy, occ_logit, mag
        return energy


# Hard occupancy threshold used at INFERENCE (rasterise).
# Pixels with sigmoid(occ) < OCC_THRESHOLD are forced to 0.
OCC_THRESHOLD = 0.55


class ResolutionInvariantSR(nn.Module):
    """Full pipeline: encode LR point cloud -> z; decode at any (eta,phi)."""

    def __init__(self, C: int = 1, latent_dim: int = 256,
                 inr_hidden: int = 256, inr_layers: int = 5):
        super().__init__()
        self.C       = C
        self.encoder = PointNetEncoder(in_features=2 + C, latent_dim=latent_dim)
        self.decoder = INRDecoder(latent_dim=latent_dim, hidden_dim=inr_hidden,
                                  n_layers=inr_layers, out_C=C)

    def forward(self, lr_pc, query_pts, return_parts: bool = False):
        z = self.encoder(lr_pc)
        return self.decoder(z, query_pts, return_parts=return_parts)

    @torch.no_grad()
    def rasterise(self, lr_pc, target_H, target_W,
                  occ_threshold: float = OCC_THRESHOLD):
        """Produce a full image at ANY resolution with a hard occupancy gate."""
        self.eval()
        B   = lr_pc.shape[0]
        dev = lr_pc.device
        eta = torch.linspace(-1, 1, target_H, device=dev)
        phi = torch.linspace(-1, 1, target_W, device=dev)
        ETA, PHI = torch.meshgrid(eta, phi, indexing='ij')
        grid   = (torch.stack([ETA.flatten(), PHI.flatten()], dim=-1)
                  .unsqueeze(0).expand(B, -1, -1))   # (B, H*W, 2)
        energy, occ_logit, _ = self.forward(lr_pc, grid, return_parts=True)
        # Hard gate: sparse output (the soft gate alone leaves a faint floor).
        energy = torch.where(torch.sigmoid(occ_logit) > occ_threshold,
                             energy, torch.zeros_like(energy))
        return energy.transpose(1, 2).reshape(B, self.C, target_H, target_W)


# ── Instantiate model ─────────────────────────────────────────────────────────
model = ResolutionInvariantSR(C=C, latent_dim=256, inr_hidden=256, inr_layers=5)
model = model.to(DEVICE)

# torch.compile gives ~20-30% throughput boost on CUDA with PyTorch >= 2.0.
# Skip on MPS/CPU — compatibility issues not worth the risk.
if DEVICE == 'cuda' and tuple(int(x) for x in torch.__version__.split('.')[:2]) >= (2, 0):
    try:
        model = torch.compile(model)
        print('torch.compile() applied.')
    except Exception as e:
        print(f'torch.compile() skipped: {e}')

n_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {n_params:,}')

dummy_pc    = torch.zeros(4, 512, 2 + C, device=DEVICE)
dummy_query = torch.rand(4, 256, 2, device=DEVICE) * 2 - 1
out         = model(dummy_pc, dummy_query)
e, occ, mag = model(dummy_pc, dummy_query, return_parts=True)
print(f'Forward OK — energy {tuple(out.shape)}, occ {tuple(occ.shape)}, '
      f'mag {tuple(mag.shape)}')
for res in [64, 125, 256]:
    shape = tuple(model.rasterise(dummy_pc, res, res).shape)
    print(f'  rasterise {res}x{res}: {shape}')


In [ ]:
class PhysicsAwareLoss(nn.Module):
    """
    Three components:
      occupancy BCE  — where is energy? (key term for sparse output)
      weighted L1    — magnitude error, upweighting active pixels
      energy cons.   — bounded |sum_pred - sum_gt|
    """

    def __init__(self, lambda_rec: float = 1.0, lambda_energy: float = 1.0,
                 lambda_occ: float = 3.0, nonzero_weight: float = 20.0,
                 occ_pos_weight: float = 1.0,
                 zero_thresh: float = ZERO_SUPPRESSION_THRESHOLD):
        super().__init__()
        self.lam_rec    = lambda_rec
        self.lam_energy = lambda_energy
        self.lam_occ    = lambda_occ
        self.alpha      = nonzero_weight
        self.occ_pos_w  = occ_pos_weight
        self.thresh     = zero_thresh

    def forward(self, E_pred, E_gt, occ_logit=None):
        target_occ = (E_gt > self.thresh).float()

        if occ_logit is not None:
            pw    = torch.tensor(self.occ_pos_w, device=occ_logit.device)
            l_occ = F.binary_cross_entropy_with_logits(occ_logit, target_occ, pos_weight=pw)
        else:
            l_occ = torch.zeros((), device=E_pred.device)

        # Active-pixel L1 + small all-pixel regularisation.
        active       = target_occ
        n_act        = active.sum().clamp(min=1)
        l_rec_active = ((E_pred - E_gt).abs() * active).sum() / n_act
        l_rec_all    = (E_pred - E_gt).abs().mean()
        l_rec        = l_rec_active + 0.1 * l_rec_all

        E_sum_pred = E_pred.sum(dim=(1, 2))
        E_sum_gt   = E_gt.sum(dim=(1, 2))
        l_energy   = (E_sum_pred - E_sum_gt).abs().mean()

        total = (self.lam_rec * l_rec + self.lam_energy * l_energy +
                 self.lam_occ * l_occ)

        with torch.no_grad():
            response = (E_sum_pred / (E_sum_gt + 1e-6)).clamp(0, 10).mean()
            if occ_logit is not None:
                pred_hit  = (torch.sigmoid(occ_logit) > 0.5).float()
                tp        = (pred_hit * target_occ).sum()
                recall    = tp / target_occ.sum().clamp(min=1)
                precision = tp / pred_hit.sum().clamp(min=1)
            else:
                recall = precision = torch.zeros(())

        return {
            'loss':      total,
            'l_rec':     l_rec.item(),
            'l_energy':  l_energy.item(),
            'l_occ':     l_occ.item(),
            'response':  response.item(),
            'recall':    float(recall),
            'precision': float(precision),
        }


criterion = PhysicsAwareLoss(lambda_rec=1.0, lambda_energy=0.2, lambda_occ=3.0,
                              nonzero_weight=10.0, occ_pos_weight=2.0)
print('Loss function OK  (gated: BCE occupancy + weighted L1 + energy conservation)')


In [ ]:
EPOCHS    = 10       # increase to 50-100 for production training
LR        = 1e-4
CLIP_GRAD = 1.0
LOG_EVERY = 1
WARMUP    = 4        # linear LR warmup (stabilises SIREN init)

# Two param groups: occupancy net learns at 10x higher LR than SIREN trunk.
# SIREN's small init + omega_0=30 needs low LR; decoupling prevents the
# occ BCE from stalling at ln2.
occ_params, base_params = [], []
for name, p in model.named_parameters():
    (occ_params if 'occ_net' in name else base_params).append(p)

optimizer = torch.optim.AdamW([
    {'params': base_params, 'lr': LR},
    {'params': occ_params,  'lr': 1e-3},
], weight_decay=1e-4)


def lr_lambda(epoch):
    if epoch < WARMUP:
        return (epoch + 1) / WARMUP
    prog = (epoch - WARMUP) / max(1, EPOCHS - WARMUP)
    return 0.5 * (1 + math.cos(math.pi * prog))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

history = {k: [] for k in ['train_loss', 'val_loss', 'val_response',
                            'train_l_rec', 'train_l_occ', 'val_l_occ',
                            'val_recall',  'val_precision']}


def run_epoch(loader, train: bool, desc: str = '') -> dict:
    model.train(train)
    totals = {k: 0.0 for k in ['loss', 'l_rec', 'l_energy', 'l_occ',
                                 'response', 'recall', 'precision']}
    n    = len(loader)
    pbar = tqdm(loader, desc=desc, leave=False, total=n)

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in pbar:
            lr_pc     = batch['lr_pc'].to(DEVICE, non_blocking=True)
            query_pts = batch['query_pts'].to(DEVICE, non_blocking=True)
            query_e   = batch['query_e'].to(DEVICE, non_blocking=True)

            # Mixed precision forward + loss.
            if USE_AMP:
                with torch.autocast(device_type='cuda'):
                    E_pred, occ_logit, _ = model(lr_pc, query_pts, return_parts=True)
                    result = criterion(E_pred, query_e, occ_logit)
            else:
                E_pred, occ_logit, _ = model(lr_pc, query_pts, return_parts=True)
                result = criterion(E_pred, query_e, occ_logit)

            if train:
                # set_to_none=True skips zeroing tensor memory (~5% faster).
                optimizer.zero_grad(set_to_none=True)
                if USE_AMP:
                    SCALER.scale(result['loss']).backward()
                    SCALER.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
                    SCALER.step(optimizer)
                    SCALER.update()
                else:
                    result['loss'].backward()
                    nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
                    optimizer.step()

            for k in totals:
                v = result[k]
                totals[k] += float(v) if isinstance(v, float) else v.item()
            pbar.set_postfix(loss=f"{result['loss'].item():.4f}")

    return {k: v / n for k, v in totals.items()}


# ── Checkpoint path ───────────────────────────────────────────────────────────
CKPT_PATH = os.path.join(CKPT_DIR, 'best_sr_model.pt')
assert next(model.parameters()).device.type == DEVICE, \
    f'Model is on wrong device (expected {DEVICE})'
print(f'Training on {DEVICE}  |  {len(train_loader)} batches/epoch  '
      f'|  checkpoint -> {CKPT_PATH}')
print(f'AMP={USE_AMP}  workers={NUM_WORKERS}  pin_memory={PIN_MEMORY}')

best_val_loss = float('inf')
for epoch in range(1, EPOCHS + 1):
    t0        = time.time()
    t_metrics = run_epoch(train_loader, train=True,  desc=f'E{epoch}/{EPOCHS} train')
    v_metrics = run_epoch(val_loader,   train=False, desc=f'E{epoch}/{EPOCHS} val')
    scheduler.step()
    dt = time.time() - t0

    history['train_loss'].append(t_metrics['loss'])
    history['val_loss'].append(v_metrics['loss'])
    history['val_response'].append(v_metrics['response'])
    history['train_l_rec'].append(t_metrics['l_rec'])
    history['train_l_occ'].append(t_metrics['l_occ'])
    history['val_l_occ'].append(v_metrics['l_occ'])
    history['val_recall'].append(v_metrics['recall'])
    history['val_precision'].append(v_metrics['precision'])

    star = ''
    if v_metrics['loss'] < best_val_loss:
        best_val_loss = v_metrics['loss']
        # Robust checkpoint: save optimizer + scheduler state for resuming.
        torch.save({
            'model':     model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'epoch':     epoch,
            'val_loss':  best_val_loss,
        }, CKPT_PATH)
        star = ' *'

    if epoch % LOG_EVERY == 0 or epoch == 1 or epoch == EPOCHS:
        print(f'Ep {epoch:3d}/{EPOCHS}  '
              f'train={t_metrics["loss"]:.4f}  val={v_metrics["loss"]:.4f}  '
              f'l_rec={t_metrics["l_rec"]:.4f}  l_occ={t_metrics["l_occ"]:.4f}  '
              f'resp={v_metrics["response"]:.3f}  recall={v_metrics["recall"]:.3f}  '
              f'lr={scheduler.get_last_lr()[0]:.1e}  {dt:.0f}s{star}')

print(f'\nBest val loss: {best_val_loss:.5f}  |  saved to {CKPT_PATH}')

# Memory cleanup after training.
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
gc.collect()


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 4))
fig.suptitle('Training Curves', fontsize=14)

axes[0].plot(history['train_loss'], label='Train', color='#e84393')
axes[0].plot(history['val_loss'],   label='Val',   color='#00b4d8')
axes[0].set_yscale('log')
axes[0].set_title('Total Loss (log)'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history['train_l_rec'], label='L1 magnitude',      color='#e84393')
axes[1].plot(history['train_l_occ'], label='Occ BCE (train)',   color='#06d6a0')
axes[1].plot(history['val_l_occ'],   label='Occ BCE (val)',     color='#118ab2', ls='--')
axes[1].axhline(0.6931, color='gray', ls=':', lw=1, label='ln2 (chance)')
axes[1].set_title('Loss Components'); axes[1].set_xlabel('Epoch'); axes[1].legend(fontsize=7)

axes[2].plot(history['val_recall'],    label='Recall',    color='#ff9f1c')
axes[2].plot(history['val_precision'], label='Precision', color='#9b5de5')
axes[2].set_ylim(0, 1)
axes[2].set_title('Occupancy Recall / Precision (Val)')
axes[2].set_xlabel('Epoch'); axes[2].legend()

axes[3].plot(history['val_response'], color='#ff9f1c', lw=2)
axes[3].axhline(1.0, color='gray', ls='--', label='Ideal = 1.0')
axes[3].set_title('Energy Response (Val)'); axes[3].set_xlabel('Epoch'); axes[3].legend()

plt.tight_layout()
savefig(fig, 'training_curves')
plt.show()

print(f"Final | occ_BCE(val)={history['val_l_occ'][-1]:.3f}  "
      f"recall={history['val_recall'][-1]:.3f}  "
      f"precision={history['val_precision'][-1]:.3f}  "
      f"response={history['val_response'][-1]:.3f}")
if history['val_l_occ'][-1] > 0.65:
    print('  !! Occupancy BCE near ln2 (0.693) — gate not learning. '
          'Try raising occ_net LR or lambda_occ.')


In [ ]:
# ── Load best checkpoint (robust: loads optimizer+scheduler too if present) ───
def _try_load_checkpoint(path):
    if not os.path.exists(path):
        print(f'No checkpoint at {path} — using in-memory trained model.')
        return False
    ckpt = torch.load(path, map_location=DEVICE)
    try:
        # Use non-strict load if we detect a compiled-model prefix mismatch.
        state = ckpt['model']
        try:
            model.load_state_dict(state, strict=True)
        except RuntimeError:
            # torch.compile wraps keys with '_orig_mod.' prefix; strip it.
            state = {k.replace('_orig_mod.', ''): v for k, v in state.items()}
            model.load_state_dict(state, strict=True)
        # Restore optimizer and scheduler if available.
        if 'optimizer' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer'])
        if 'scheduler' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler'])
        model.eval()
        print(f'Loaded checkpoint  epoch={ckpt.get("epoch","?")}  '
              f'val_loss={ckpt.get("val_loss", float("nan")):.5f}')
        return True
    except Exception as e:
        print(f'Checkpoint mismatch — keeping in-memory model.  ({type(e).__name__}: {e})')
        return False

_try_load_checkpoint(CKPT_PATH)
model.eval()

# ── Occupancy diagnostic ──────────────────────────────────────────────────────
with torch.no_grad():
    s  = val_dataset[0]
    pc = s['lr_pc'].unsqueeze(0).to(DEVICE, non_blocking=True)
    rq = torch.rand(1, 2000, 2, device=DEVICE) * 2 - 1
    _, occ_logit, mag = model(pc, rq, return_parts=True)
    occ = torch.sigmoid(occ_logit)
    print(f'Occupancy on 2000 random queries:')
    print(f'  sigmoid(occ): min={occ.min():.3f} mean={occ.mean():.3f} '
          f'max={occ.max():.3f}')
    print(f'  frac > OCC_THRESHOLD ({OCC_THRESHOLD}): '
          f'{(occ > OCC_THRESHOLD).float().mean():.3%}')
    print(f'  magnitude:    mean={mag.mean():.4f} max={mag.max():.4f}')
    img_diag = model.rasterise(pc, *HR_RES).squeeze(0).cpu().numpy()
    print(f'  rasterised {HR_RES}: sum={img_diag.sum():.3f}  '
          f'active(>1e-3)={int((img_diag > 1e-3).sum())}/{img_diag.size}')
    if img_diag.sum() < 1e-3:
        print('  !! OUTPUT BLACK — lower OCC_THRESHOLD or train more epochs.')


def compute_psnr(pred, gt, max_val=1.0):
    mse = np.mean((pred - gt) ** 2)
    return 20 * np.log10(max_val / (np.sqrt(mse) + 1e-10))

def compute_energy_response(pred, gt):
    return pred.sum() / (gt.sum() + 1e-8)

def bicubic_upsample(lr, target_H, target_W):
    t  = torch.from_numpy(lr).unsqueeze(0)
    up = F.interpolate(t, size=(target_H, target_W),
                       mode='bicubic', align_corners=False)
    return up.squeeze(0).numpy().clip(0, None)


# ── Batched evaluation (groups BATCH_SIZE samples) ───────────────────────────
N_EVAL   = min(64, len(val_dataset))
metrics  = {k: [] for k in ['psnr_bicubic', 'psnr_model',
                              'response_bicubic', 'response_model',
                              'l1_bicubic',   'l1_model']}
HR_H, HR_W = HR_RES

with torch.no_grad():
    # Process in batches to match training throughput.
    for batch_start in tqdm(range(0, N_EVAL, BATCH_SIZE),
                             desc='Evaluating', leave=True):
        batch_end     = min(batch_start + BATCH_SIZE, N_EVAL)
        batch_indices = list(range(batch_start, batch_end))
        samples       = [val_dataset[i] for i in batch_indices]

        lr_pc_batch = torch.stack([s['lr_pc'] for s in samples]).to(DEVICE, non_blocking=True)
        sr_batch    = model.rasterise(lr_pc_batch, HR_H, HR_W).cpu().numpy()   # (B, C, H, W)

        for j, sample in enumerate(samples):
            hr_gt  = sample['hr_img'].numpy()
            lr_img = sample['lr_img'].numpy()
            bic    = bicubic_upsample(lr_img, HR_H, HR_W)
            sr     = sr_batch[j]

            metrics['psnr_bicubic'].append(compute_psnr(bic, hr_gt))
            metrics['psnr_model'].append(compute_psnr(sr, hr_gt))
            metrics['response_bicubic'].append(compute_energy_response(bic, hr_gt))
            metrics['response_model'].append(compute_energy_response(sr, hr_gt))
            metrics['l1_bicubic'].append(np.abs(bic - hr_gt).mean())
            metrics['l1_model'].append(np.abs(sr - hr_gt).mean())

print('\n=== Evaluation Results ===')
print(f'{"Metric":25s}  {"Bicubic":>10s}  {"INR-SR (ours)":>14s}')
print('-' * 56)
for m_bic, m_mod, label in [
    (metrics['psnr_bicubic'],     metrics['psnr_model'],     'PSNR (dB) up'),
    (metrics['response_bicubic'], metrics['response_model'], 'Energy Response ->1.0'),
    (metrics['l1_bicubic'],       metrics['l1_model'],       'L1 Error down'),
]:
    print(f'{label:25s}  {np.mean(m_bic):10.4f}  {np.mean(m_mod):14.4f}')


In [ ]:
N_VIS = 3
fig   = plt.figure(figsize=(20, 5 * N_VIS))
fig.suptitle(f'Super-Resolution Comparison  (OCC_THRESHOLD={OCC_THRESHOLD})',
             fontsize=14, fontweight='bold')
col_titles = ['LR Input (64x64)', 'Bicubic (125x125)',
              'INR-SR (125x125)', 'INR-SR (256x256) <- res-invariant!',
              'HR Ground Truth (125x125)']

with torch.no_grad():
    for row in range(N_VIS):
        sample  = val_dataset[row]
        lr_pc   = sample['lr_pc'].unsqueeze(0).to(DEVICE, non_blocking=True)
        hr_gt   = sample['hr_img'].numpy()
        lr_img  = sample['lr_img'].numpy()
        bic_sr  = bicubic_upsample(lr_img, *HR_RES)
        inr_sr  = model.rasterise(lr_pc, *HR_RES).squeeze(0).cpu().numpy()
        inr_sr2 = model.rasterise(lr_pc, 256, 256).squeeze(0).cpu().numpy()
        images_row = [lr_img, bic_sr, inr_sr, inr_sr2, hr_gt]

        for col, (arr, title) in enumerate(zip(images_row, col_titles)):
            ax   = fig.add_subplot(N_VIS, 5, row * 5 + col + 1)
            a    = arr[0]
            vmax = max(float(a.max()), 1e-3)
            vmin = max(vmax * 1e-3, 1e-6)
            ax.imshow(np.clip(a, vmin, None), cmap='hot',
                      norm=LogNorm(vmin=vmin, vmax=vmax), aspect='equal')
            if row == 0:
                ax.set_title(title, fontsize=8, fontweight='bold')
            ax.set_xlabel(f'E={arr.sum():.3f}  active={int((a>1e-3).sum())}',
                          fontsize=7)
            ax.axis('off')

plt.tight_layout()
savefig(fig, 'sr_comparison')
plt.show()


In [ ]:
sample  = val_dataset[0]
lr_pc   = sample['lr_pc'].unsqueeze(0).to(DEVICE, non_blocking=True)

resolutions = [32, 48, 64, 96, 125, 192, 256]
fig, axes = plt.subplots(1, len(resolutions), figsize=(21, 4))
fig.suptitle('Resolution Invariance: Same Model, Any Output Resolution', fontsize=13)

with torch.no_grad():
    for ax, r in zip(axes, resolutions):
        sr      = model.rasterise(lr_pc, r, r).squeeze(0).cpu().numpy()
        display = np.clip(sr[0], 1e-6, None)
        ax.imshow(display, cmap='hot',
                  norm=LogNorm(vmin=1e-4, vmax=1.0), aspect='equal')
        ax.set_title(f'{r}x{r}\nE={sr.sum():.3f}', fontsize=8)
        ax.axis('off')

plt.tight_layout()
savefig(fig, 'resolution_invariance')
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Physics Validation Metrics', fontsize=14)

axes[0].hist(metrics['response_bicubic'], bins=30, alpha=0.6, color='#00b4d8',
             label=f'Bicubic  mu={np.mean(metrics["response_bicubic"]):.3f}')
axes[0].hist(metrics['response_model'],   bins=30, alpha=0.6, color='#e84393',
             label=f'INR-SR   mu={np.mean(metrics["response_model"]):.3f}')
axes[0].axvline(1.0, color='k', ls='--', lw=2, label='Ideal = 1.0')
axes[0].set_xlabel('Energy Response  sum(E_pred) / sum(E_gt)')
axes[0].set_ylabel('Count')
axes[0].set_title('Energy Response Distribution')
axes[0].legend(fontsize=8)

axes[1].hist(metrics['psnr_bicubic'], bins=25, alpha=0.6, color='#00b4d8',
             label=f'Bicubic  mu={np.mean(metrics["psnr_bicubic"]):.2f} dB')
axes[1].hist(metrics['psnr_model'],   bins=25, alpha=0.6, color='#e84393',
             label=f'INR-SR   mu={np.mean(metrics["psnr_model"]):.2f} dB')
axes[1].set_xlabel('PSNR (dB)'); axes[1].set_ylabel('Count')
axes[1].set_title('PSNR Distribution'); axes[1].legend(fontsize=8)

axes[2].scatter(metrics['response_bicubic'], metrics['psnr_bicubic'],
                alpha=0.5, s=15, color='#00b4d8', label='Bicubic')
axes[2].scatter(metrics['response_model'],   metrics['psnr_model'],
                alpha=0.5, s=15, color='#e84393', label='INR-SR')
axes[2].axvline(1.0, color='k', ls='--', lw=1)
axes[2].set_xlabel('Energy Response'); axes[2].set_ylabel('PSNR (dB)')
axes[2].set_title('Energy Response vs PSNR')
axes[2].legend(fontsize=8)

plt.tight_layout()
savefig(fig, 'physics_validation')
plt.show()


In [ ]:
@torch.no_grad()
def predict_million_points(lr_pc: torch.Tensor,
                            n_points: int = 1_000_000,
                            chunk_size: int = 50_000) -> tuple:
    """
    Query the INR at n_points random (eta, phi) coordinates.
    Memory-efficient: encodes once, decodes in chunks.
    Returns (coords, energies) as numpy float32 arrays.
    """
    model.eval()
    dev         = lr_pc.device
    all_coords  = torch.rand(n_points, 2, device=dev) * 2 - 1   # (n_points, 2)
    all_energies = []

    z = model.encoder(lr_pc)   # encode once — (1, latent_dim)

    for start in tqdm(range(0, n_points, chunk_size),
                      desc='Decoding chunks', leave=False):
        end    = min(start + chunk_size, n_points)
        coords = all_coords[start:end].unsqueeze(0)   # (1, chunk, 2)
        E_pred = model.decoder(z, coords)             # (1, chunk, C)
        all_energies.append(E_pred.squeeze(0).cpu())

    all_energies = torch.cat(all_energies, dim=0)   # (n_points, C)
    return all_coords.cpu().numpy(), all_energies.numpy()


sample    = val_dataset[0]
lr_pc_t   = sample['lr_pc'].unsqueeze(0).to(DEVICE, non_blocking=True)

print('Predicting 1,000,000 points...')
coords_1m, energies_1m = predict_million_points(lr_pc_t, n_points=1_000_000)

active_mask     = energies_1m[:, 0] > ZERO_SUPPRESSION_THRESHOLD
active_coords   = coords_1m[active_mask]
active_energies = energies_1m[active_mask, 0]

print(f'Total points     : {len(coords_1m):,}')
print(f'Active after ZS  : {active_mask.sum():,}  ({active_mask.mean():.2%})')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('1 Million Point Prediction with Zero Suppression', fontsize=13)

n_show = min(5000, len(active_coords))
vmax   = float(active_energies.max()) if len(active_energies) > 0 else 1.0
sc = axes[0].scatter(active_coords[:n_show, 1], active_coords[:n_show, 0],
                     c=active_energies[:n_show], cmap='hot', s=0.5,
                     norm=LogNorm(vmin=1e-4, vmax=vmax))
axes[0].set_xlim(-1, 1); axes[0].set_ylim(-1, 1); axes[0].invert_yaxis()
axes[0].set_title(f'Active Points ({n_show:,} shown, 1M predicted)')
axes[0].set_xlabel('phi'); axes[0].set_ylabel('eta')
plt.colorbar(sc, ax=axes[0], label='Predicted E')

# Rasterise all 1M active points to 512x512.
if len(active_coords) > 0:
    hr_pc_arr = np.concatenate([active_coords, active_energies[:, None]], axis=-1)
    img_512   = pointcloud_to_image(hr_pc_arr, 512, 512, C=1, aggregation='sum')
    axes[1].imshow(np.clip(img_512[0], 1e-6, None), cmap='hot',
                   norm=LogNorm(vmin=1e-4, vmax=1.0), aspect='equal')
    axes[1].set_title('1M points rasterised to 512x512 (ultra-high resolution)')
else:
    axes[1].set_title('No active points (model may need more training)')
axes[1].axis('off')

plt.tight_layout()
savefig(fig, '1M_point_prediction')
plt.show()

if DEVICE == 'cuda':
    torch.cuda.empty_cache()
gc.collect()


In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('=' * 64)
print('  CMS CALORIMETER SR — RESULTS SUMMARY')
print('  GSoC 2026 | ML4Sci | Rajveer Rathod')
print('=' * 64)
print(f'\n  Dataset:  {N:,} jets  C={C}  orig {CELL_SHAPE[1]}x{CELL_SHAPE[2]}')
print(f'  LR res:   {LR_RES}  ->  HR target: {HR_RES}')
print(f'  Model:    PointNet encoder + gated SIREN-INR decoder')
print(f'            (occupancy classification x magnitude regression)')
print(f'  Params:   {trainable:,}')
print(f'  AMP:      {USE_AMP}  |  workers: {NUM_WORKERS}  |  compile: '
      f'{"yes (CUDA)" if DEVICE == "cuda" else "no"}')

print(f'\n  {"Metric":30s}  {"Bicubic":>10s}  {"INR-SR":>10s}  Delta')
print('  ' + '-' * 60)

b_psnr = np.mean(metrics['psnr_bicubic'])
m_psnr = np.mean(metrics['psnr_model'])
b_resp = np.mean(metrics['response_bicubic'])
m_resp = np.mean(metrics['response_model'])
b_l1   = np.mean(metrics['l1_bicubic'])
m_l1   = np.mean(metrics['l1_model'])

def delta(a, b, higher_is_better=True):
    d = b - a
    arrow = 'up' if (d > 0) == higher_is_better else 'dn'
    return f'{arrow} {abs(d):.4f}'

print(f'  {"PSNR (dB) up":30s}  {b_psnr:10.3f}  {m_psnr:10.3f}  '
      f'{delta(b_psnr, m_psnr, True)}')
print(f'  {"Energy Response (->1.0)":30s}  {b_resp:10.4f}  {m_resp:10.4f}  '
      f'{delta(abs(b_resp-1), abs(m_resp-1), False)}')
print(f'  {"L1 Error down":30s}  {b_l1:10.5f}  {m_l1:10.5f}  '
      f'{delta(b_l1, m_l1, False)}')

print(f'\n  Occupancy gate:')
print(f'    BCE (val)   {history["val_l_occ"][-1]:.4f}  (chance=0.693)')
print(f'    Recall      {history["val_recall"][-1]:.4f}')
print(f'    Precision   {history["val_precision"][-1]:.4f}')

print(f'\n  Resolution invariant : yes (tested 32 to 512 px)')
try:
    print(f'  1M point prediction  : yes ({active_mask.mean():.1%} active after ZS)')
except NameError:
    print('  1M point prediction  : run cell 22 first')
print(f'  Checkpoint saved     : {CKPT_PATH}')
print('=' * 64)


In [ ]:
# ── Optional: push checkpoint to HuggingFace Hub ─────────────────────────────
# Uncomment and set your HF token to push.

# from huggingface_hub import HfApi
# api = HfApi(token='YOUR_HF_TOKEN')
# api.upload_file(
#     path_or_fileobj=CKPT_PATH,
#     path_in_repo='checkpoints/best_sr_model.pt',
#     repo_id='YOUR_HF_USERNAME/cms-calorimeter-superres-inr',
#     repo_type='model',
# )
# print('Pushed to HuggingFace Hub.')

print(f'Notebook complete.')
print(f'Checkpoint : {CKPT_PATH}')
print(f'Figures    : {FIGURES_DIR}')
print(f'Cache      : {CACHE_DIR}')
